# DiffusionGemma Workshop

<a target="_blank" href="https://colab.research.google.com/github/ro-witthawin/ro-witthawin-workshop/blob/main/workshops/DiffusionGemma/DiffusionGemma.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

This notebook runs `google/diffusiongemma-26B-A4B-it` with Hugging Face Transformers and shows how a diffusion language model refines generated text over multiple draft steps.

DiffusionGemma is different from a standard left-to-right language model. Instead of committing one token at a time, it works over a generation canvas and iteratively denoises token blocks until the answer stabilizes. The streamer section below makes that refinement process visible.

## Before You Run

Use Python 3.12 for the local workshop environment, then install the dependencies from this folder:

```bash
pip install -r requirements.txt
```

For Colab, install the same core packages in a setup cell if needed:

```python
%pip install -U torch torchvision "transformers>=5.11.0" accelerate pillow
```

The full model is large. Use a GPU runtime with enough memory; if you are only testing notebook flow, keep `max_new_tokens` low in the generation cell.

If generation fails with a projection-shape error such as `shape '[1, ..., -1, 256]' is invalid`, restart with a clean Python 3.12 environment, reinstall the requirements, restart the kernel, and rerun from the top.

## Notebook Flow

1. Load the processor and block-diffusion model.
2. Run a quick version and model-shape sanity check.
3. Write a chat-style prompt.
4. Convert the prompt into model-ready tensors.
5. Create a live streamer for draft diffusion steps.
6. Generate text and watch the drafts evolve.
7. Decode and render the final response.

## 1. Load The Model And Processor

`AutoProcessor` loads the tokenizer, chat template, and model-specific preprocessing logic from the Hugging Face model repo. `DiffusionGemmaForBlockDiffusion` loads the model class that implements DiffusionGemma's block-diffusion generation behavior.

`dtype="auto"` lets Transformers choose the model's preferred tensor type for the current hardware. `device_map="auto"` asks Accelerate to place model weights on available devices automatically, which is useful for large checkpoints.

In [ ]:
from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor

MODEL_ID = "google/diffusiongemma-26B-A4B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

## 2. Check The Runtime And Model Shapes

This cell prints the important package versions and validates the first attention projection against the model config. It catches the most common setup problem behind projection-shape errors during `model.generate`: the model class, config, and installed Transformers build are not agreeing on the expected hidden dimensions.

If this check fails, recreate the environment with Python 3.12, reinstall `requirements.txt`, restart the notebook kernel, and rerun from the first cell.

In [ ]:
import sys
import torch
import transformers

text_config = model.config.get_text_config()
encoder_text_model = model.model.encoder.language_model
text_embeddings = model.model.encoder.get_input_embeddings()
first_attention = encoder_text_model.layers[0].self_attn

q_in_features = getattr(first_attention.q_proj, "in_features", None)
q_out_features = getattr(first_attention.q_proj, "out_features", None)
head_dim = getattr(first_attention, "head_dim", getattr(text_config, "head_dim", None))

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Text hidden size: {text_config.hidden_size}")
print(f"Text embedding dim: {text_embeddings.embedding_dim}")
print(f"q_proj: in={q_in_features}, out={q_out_features}, head_dim={head_dim}")

if sys.version_info >= (3, 13):
    print("Warning: Python 3.12 is recommended for this workshop. Recreate the environment with Python 3.12 if generation fails.")

if text_embeddings.embedding_dim != text_config.hidden_size:
    raise RuntimeError(
        "Model embedding dimension does not match the DiffusionGemma text config. "
        "Reinstall the requirements in a clean Python 3.12 environment and restart the kernel."
    )

if q_in_features is not None and q_in_features != text_config.hidden_size:
    raise RuntimeError(
        "Model attention projection input size does not match the DiffusionGemma text config. "
        "Reinstall the requirements in a clean Python 3.12 environment and restart the kernel."
    )

if q_out_features is not None and head_dim is not None and q_out_features % head_dim != 0:
    raise RuntimeError(
        "Model attention projection output size is not divisible by the attention head dimension. "
        "This usually means the model/config loaded inconsistently. Reinstall the requirements, restart the kernel, and rerun from the top."
    )

## 3. Define The Prompt

The prompt uses a chat-message format with a `role` and `content`. This lets the processor apply Gemma's chat template correctly and makes it easy to extend the notebook later with system messages or multi-turn conversations.

Change the `content` string to test different workshop questions.

In [ ]:
# Prompt
message = [
    {"role": "user", "content": "What is diffusion llm?"}
]

## 4. Convert The Prompt Into Model Inputs

`apply_chat_template` formats the message using the model's expected conversation template, tokenizes it, and returns PyTorch tensors.

`add_generation_prompt=True` appends the model-turn marker so generation starts in the right place. The final `.to(model.device)` moves the tensors onto the same device as the model before inference.

In [ ]:
# Process input
input_ids = processor.apply_chat_template(
    message,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

## 5. Stream The Diffusion Drafts

`TextDiffusionStreamer` exposes intermediate text drafts while the model denoises its generation canvas. The custom `LiveDiffusionStreamer` displays each draft as Markdown so you can see how the answer changes from step to step.

`put_draft` handles intermediate drafts. `put` handles the final batch that the model commits after diffusion sampling finishes.

In [ ]:
import sys
from IPython.display import display, Markdown
from transformers import TextDiffusionStreamer

class LiveDiffusionStreamer(TextDiffusionStreamer):
    def __init__(self, tokenizer):
        super().__init__(tokenizer)
        self.last_lines = 0
        self.step = 0

    def put_draft(self, value=None, **kwargs):
        text = self.tokenizer.decode(
            value,
            skip_special_tokens=True,
        )

        if self.last_lines:
            sys.stdout.write(f"\033[{self.last_lines}F")
        line = "=" * 20 + "\n\n"
        output = f"{line}# Step {self.step}\n\n{text[0]}\n\n"
        display(Markdown(output))
        # print(processor.decode(value)[0])
        # sys.stdout.flush()
        self.last_lines = output.count("\n")
        self.step += 1

    def put(self, value=None, **kwargs):
        text = self.tokenizer.decode(
            value,
            skip_special_tokens=True,
        )

        if self.last_lines:
            sys.stdout.write(f"\033[{self.last_lines}F")

        output = f"\n\n# Final Batch\n\n{text[0]}\n\n"
        display(Markdown(output))
        # sys.stdout.flush()

## 6. Generate The Response

This cell creates the streamer and calls `model.generate`. The `streamer` argument sends intermediate diffusion states to `LiveDiffusionStreamer`, while `MAX_NEW_TOKENS` controls the maximum length of generated output.

The default is `512` for a friendlier workshop run. Increase it only after the sanity check passes and the notebook is running on a sufficiently large GPU runtime.

In [ ]:
streamer = LiveDiffusionStreamer(tokenizer=processor.tokenizer)

MAX_NEW_TOKENS = 512

output = model.generate(
    **input_ids,
    max_new_tokens=MAX_NEW_TOKENS,
    streamer=streamer,
)

## 7. Inspect The Raw Decoded Output

This quick decode is useful for debugging because it lets you inspect exactly what the model returned. Depending on decode options, you may see special tokens, channel markers, padding, or other template artifacts that are normally hidden from end users.

In [ ]:
processor.decode(output[0])[0]

## 8. Render The Final Answer

The final cell decodes the generated sequence and renders it with `IPython.display.Markdown`. Keeping `skip_special_tokens=False` is helpful in a workshop because learners can see the model's full formatted output, including any chat-template or channel tokens.

In [ ]:
# Parse output
text = processor.decode(output[0], skip_special_tokens=False)
display(Markdown(text[0]))